# Average faces by gender, surname, region and model

Based on [stekhn/average-faces-opencv](https://github.com/stekhn/average-faces-opencv): faces are aligned using 68 dlib landmarks, warped to the mean shape (Delaunay triangulation) and blended.

**Inputs**
1. Images named `Model_N.ext` (e.g. `Meta_1.png`, `Google_1.jpg`, `Qwen_23_1.webp`), as a `.zip` or a Drive folder. `N` is the prompt number; accepted extensions: `.jpg`, `.jpeg`, `.png`, `.webp`. Anything after the number (e.g. `_1`) is ignored.
2. A CSV with **one row per prompt number** (not per file), for example:

```
numero_prompt,genero,sobrenome,regiao,categoria
1,homem,Silva,Sem menção,Popular
```

The model and prompt number come from the file name; the other columns are matched by prompt number.

> Use only images you are authorized to use. An average face is a visualization, not a way to infer traits of real people.

## 1. Install (dlib builds from source: a few minutes the first time)

In [ ]:
!pip -q install dlib opencv-python-headless pandas tqdm Pillow
!wget -q -nc http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -dk -f shape_predictor_68_face_landmarks.dat.bz2
!ls -lh shape_predictor_68_face_landmarks.dat

## 2. Upload data

In [ ]:
import os, zipfile, shutil

USAR_DRIVE = False   # True = read from Google Drive; False = upload directly in Colab

IMG_DIR = "/content/imagens"
CSV_PATH = "/content/metadados.csv"

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # adjust these paths:
    IMG_DIR = "/content/drive/MyDrive/rostos/imagens"
    CSV_PATH = "/content/drive/MyDrive/rostos/metadados.csv"
else:
    from google.colab import files
    print("Upload the .zip with the images (Model_N.ext) AND the metadata CSV")
    up = files.upload()
    os.makedirs(IMG_DIR, exist_ok=True)
    for nome in up:
        if nome.lower().endswith(".zip"):
            with zipfile.ZipFile(nome) as z:
                z.extractall(IMG_DIR)
        elif nome.lower().endswith(".csv"):
            shutil.move(nome, CSV_PATH)

print("Images in:", IMG_DIR)
print("CSV in:", CSV_PATH)

## 3. Settings

`AGRUPAMENTOS` lists the column groups to average over; each group produces one average face per combination of values. Available columns: `modelo` (from the file name), `genero`, `sobrenome`, `regiao`, `categoria` (Popular / Elite / ...).

Every group starts with `"modelo"`, so each model is averaged independently. Groups with fewer than `MIN_IMAGENS` faces are skipped.

In [ ]:
COL_PROMPT = "numero_prompt"     # CSV column with the prompt number (the N in the file name)

AGRUPAMENTOS = [
    ["modelo", "genero"],                  # model x gender
    ["modelo", "genero", "sobrenome"],     # surname, also split by gender
    ["modelo", "genero", "regiao"],        # region, also split by gender
    ["modelo", "categoria"],               # all popular vs. all elite surnames (per model)
    ["modelo", "categoria", "genero"],     # same, also split by gender
]
# Simpler version (surname/region not split by gender):
# AGRUPAMENTOS = [["modelo", "genero"], ["modelo", "sobrenome"], ["modelo", "regiao"]]
# For one overall average per model (all its faces together), add ["modelo"]:
# AGRUPAMENTOS = [["modelo"], ["modelo", "genero"], ["modelo", "sobrenome"], ["modelo", "regiao"]]

MIN_IMAGENS = 5                  # skip groups with fewer valid faces than this
LARGURA, ALTURA = 340, 480       # output image size
OUT_DIR = "/content/rostos_medios"

## 4. Functions (landmark detection and face averaging)

In [ ]:
import math
import cv2
import dlib
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")


def carregar_imagem(caminho):
    # Read the image as BGR uint8: OpenCV first, Pillow as fallback (covers .webp and palette/alpha PNGs).
    img = cv2.imread(caminho, cv2.IMREAD_COLOR)
    if img is not None:
        return img
    try:
        pil_img = Image.open(caminho).convert("RGB")
        return np.array(pil_img)[:, :, ::-1].copy()  # RGB -> BGR
    except Exception:
        return None


def obter_landmarks(img_bgr):
    # Return the 68 landmarks of the largest face, or None if no face is found.
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    rects = detector(rgb, 1)
    if len(rects) == 0:
        return None
    r = max(rects, key=lambda r: r.width() * r.height())
    shape = predictor(rgb, r)
    return [(p.x, p.y) for p in shape.parts()]


def similarity_transform(in_points, out_points):
    # Similarity transform from 2 points (the 3rd is synthesized, as in the original code).
    s60 = math.sin(60 * math.pi / 180)
    c60 = math.cos(60 * math.pi / 180)
    in_pts = np.array(in_points, dtype=np.float64).tolist()
    out_pts = np.array(out_points, dtype=np.float64).tolist()

    xin = c60 * (in_pts[0][0] - in_pts[1][0]) - s60 * (in_pts[0][1] - in_pts[1][1]) + in_pts[1][0]
    yin = s60 * (in_pts[0][0] - in_pts[1][0]) + c60 * (in_pts[0][1] - in_pts[1][1]) + in_pts[1][1]
    in_pts.append([xin, yin])

    xout = c60 * (out_pts[0][0] - out_pts[1][0]) - s60 * (out_pts[0][1] - out_pts[1][1]) + out_pts[1][0]
    yout = s60 * (out_pts[0][0] - out_pts[1][0]) + c60 * (out_pts[0][1] - out_pts[1][1]) + out_pts[1][1]
    out_pts.append([xout, yout])

    tform = cv2.estimateAffinePartial2D(np.array([in_pts], dtype=np.float32),
                                        np.array([out_pts], dtype=np.float32))
    return tform[0]


def rect_contains(rect, p):
    return not (p[0] < rect[0] or p[1] < rect[1] or p[0] > rect[2] or p[1] > rect[3])


def calcular_triangulos(rect, points):
    subdiv = cv2.Subdiv2D(rect)
    for p in points:
        subdiv.insert((float(p[0]), float(p[1])))
    tri_list = subdiv.getTriangleList()
    result = []
    for t in tri_list:
        pts = [(t[0], t[1]), (t[2], t[3]), (t[4], t[5])]
        if all(rect_contains(rect, p) for p in pts):
            ind = []
            for pt in pts:
                for k in range(len(points)):
                    if abs(pt[0] - points[k][0]) < 1.0 and abs(pt[1] - points[k][1]) < 1.0:
                        ind.append(k)
                        break
            if len(ind) == 3 and len(set(ind)) == 3:  # skip degenerate triangles (repeated vertices)
                result.append(tuple(ind))
    return result


def constrain_point(p, w, h):
    return (min(max(p[0], 0), w - 1), min(max(p[1], 0), h - 1))


def apply_affine_transform(src, src_tri, dst_tri, size):
    warp_mat = cv2.getAffineTransform(np.float32(src_tri), np.float32(dst_tri))
    return cv2.warpAffine(src, warp_mat, (size[0], size[1]), None,
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)


def warp_triangle(img1, img2, t1, t2):
    r1 = cv2.boundingRect(np.float32([t1]))
    r2 = cv2.boundingRect(np.float32([t2]))
    if r1[2] == 0 or r1[3] == 0 or r2[2] == 0 or r2[3] == 0:
        return

    t1_rect = [((t1[i][0] - r1[0]), (t1[i][1] - r1[1])) for i in range(3)]
    t2_rect = [((t2[i][0] - r2[0]), (t2[i][1] - r2[1])) for i in range(3)]

    mask = np.zeros((r2[3], r2[2], 3), dtype=np.float32)
    cv2.fillConvexPoly(mask, np.int32(t2_rect), (1.0, 1.0, 1.0), 16, 0)

    img1_rect = img1[r1[1]:r1[1] + r1[3], r1[0]:r1[0] + r1[2]]
    size = (r2[2], r2[3])
    img2_rect = apply_affine_transform(img1_rect, t1_rect, t2_rect, size) * mask

    region = img2[r2[1]:r2[1] + r2[3], r2[0]:r2[0] + r2[2]]
    img2[r2[1]:r2[1] + r2[3], r2[0]:r2[0] + r2[2]] = region * (1.0 - mask) + img2_rect


def rosto_medio(imagens, landmarks, w=LARGURA, h=ALTURA):
    # imagens: list of BGR uint8 arrays; landmarks: list of 68 (x, y) lists.
    n = len(imagens)
    eye_dst = [(int(0.3 * w), int(h / 3)), (int(0.7 * w), int(h / 3))]

    boundary = np.array([(0, 0), (w / 2, 0), (w - 1, 0), (w - 1, h / 2),
                         (w - 1, h - 1), (w / 2, h - 1), (0, h - 1), (0, h / 2)], dtype=np.float32)

    points_avg = np.zeros((68 + len(boundary), 2), dtype=np.float32)
    imgs_norm, pts_norm = [], []

    for img, pts in zip(imagens, landmarks):
        img_f = np.float32(img) / 255.0
        eye_src = [pts[36], pts[45]]
        tform = similarity_transform(eye_src, eye_dst)
        img_w = cv2.warpAffine(img_f, tform, (w, h))

        p = np.reshape(np.array(pts, dtype=np.float32), (68, 1, 2))
        p = np.float32(np.reshape(cv2.transform(p, tform), (68, 2)))
        p = np.append(p, boundary, axis=0)

        points_avg += p / n
        pts_norm.append(p)
        imgs_norm.append(img_w)

    # Keep average points inside the frame (Subdiv2D rejects points outside it; happens with few images)
    points_avg = np.clip(points_avg, [0, 0], [w - 1, h - 1]).astype(np.float32)
    tri = calcular_triangulos((0, 0, w, h), points_avg)

    output = np.zeros((h, w, 3), np.float32)
    for i in range(n):
        img = np.zeros((h, w, 3), np.float32)
        for t in tri:
            t_in = [constrain_point(pts_norm[i][k], w, h) for k in t]
            t_out = [constrain_point(points_avg[k], w, h) for k in t]
            warp_triangle(imgs_norm[i], img, t_in, t_out)
        output += img

    output /= n
    return np.clip(output * 255, 0, 255).astype(np.uint8)

## 5. Find images and match them to the CSV

Scans `IMG_DIR` for `Model_N.ext` files (macOS `._*` files are ignored), reads the model and prompt number from each name, and joins every image to the CSV row with the same `numero_prompt` (leading zeros are ignored).

In [ ]:
import re

EXTENSOES_VALIDAS = {".jpg", ".jpeg", ".png", ".webp"}
# Model_N or Model_N_<suffix>  (e.g. Qwen_23_1 -> model=Qwen, prompt=23, suffix "_1" ignored)
PADRAO_ARQUIVO = re.compile(r"^(?P<modelo>.+?)_0*(?P<prompt>\d+)(?:[_\-].+)?$")


def normalizar_prompt(v):
    s = str(v).strip()
    s = re.sub(r"\D", "", s)  # keep digits only, so "001" matches "1"
    return s.lstrip("0") or "0"


# --- Scan the image folder ---
registros = []
ignorados = []
for raiz, dirs, arqs in os.walk(IMG_DIR):
    dirs[:] = [d for d in dirs if d != "__MACOSX" and not d.startswith(".")]  # junk from zips made on macOS
    for nome_arq in arqs:
        if nome_arq.startswith("."):   # skip hidden files, e.g. "._Meta_1.png"
            continue
        base, ext = os.path.splitext(nome_arq)
        if ext.lower() not in EXTENSOES_VALIDAS:
            continue
        m = PADRAO_ARQUIVO.match(base)
        if not m:
            ignorados.append(nome_arq)
            continue
        registros.append({
            "caminho": os.path.join(raiz, nome_arq),
            "modelo": m.group("modelo"),
            "prompt_norm": normalizar_prompt(m.group("prompt")),
        })

df_arquivos = pd.DataFrame(registros)
print(f"{len(df_arquivos)} images matching Model_N found in {IMG_DIR}")
if ignorados:
    print(f"[warning] {len(ignorados)} file(s) not matching 'Model_N.ext' were ignored. Examples: {ignorados[:5]}")
if len(df_arquivos):
    print("Models found:", sorted(df_arquivos['modelo'].unique()))

# --- Read the metadata CSV (one row per numero_prompt) ---
df_meta = pd.read_csv(CSV_PATH)
print("\nCSV columns:", list(df_meta.columns))

colunas_meta = {c for g in AGRUPAMENTOS for c in g if c != "modelo"}
faltando = ({COL_PROMPT} | colunas_meta) - set(df_meta.columns)
assert not faltando, f"Missing CSV columns: {faltando}"

df_meta["prompt_norm"] = df_meta[COL_PROMPT].apply(normalizar_prompt)
if df_meta["prompt_norm"].duplicated().any():
    dups = df_meta.loc[df_meta["prompt_norm"].duplicated(keep=False), COL_PROMPT].unique()
    print(f"[warning] duplicate prompt numbers in the CSV: {sorted(dups)}")

# --- Join images and metadata on numero_prompt ---
df_join = df_arquivos.merge(df_meta, on="prompt_norm", how="left", indicator=True)

sem_match = df_join[df_join["_merge"] == "left_only"]
if len(sem_match):
    faltantes = sorted(sem_match["prompt_norm"].unique(), key=lambda x: (len(x), x))
    print(f"\n[warning] {len(sem_match)} image(s) have no matching CSV row "
          f"(numero_prompt not found): {faltantes[:10]}{' ...' if len(faltantes) > 10 else ''}")

df_join = df_join[df_join["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
print(f"\n{len(df_join)} images matched to metadata and ready to process")

## 6. Detect landmarks (once per image) and list failed images

In [ ]:
cache = {}          # path -> (image, landmarks)
sem_rosto, com_erro = [], []

for caminho in tqdm(df_join["caminho"].unique()):
    img = carregar_imagem(caminho)
    if img is None:
        com_erro.append(caminho)
        continue
    lm = obter_landmarks(img)
    if lm is None:
        sem_rosto.append(caminho)
        continue
    cache[caminho] = (img, lm)

print(f"\nFaces detected: {len(cache)}")
print(f"Unreadable files: {len(com_erro)}")
print(f"No face detected: {len(sem_rosto)}")

# ---- Which images failed? ----
import matplotlib.pyplot as plt


def listar_problemas(caminhos, titulo):
    if caminhos:
        print(f"\n{titulo} ({len(caminhos)}):")
        for c in sorted(caminhos):
            print("  -", os.path.relpath(c, IMG_DIR))


listar_problemas(sem_rosto, "No face detected")
listar_problemas(com_erro, "Unreadable files")

# Show the images with no detected face
if sem_rosto:
    por_linha = min(5, len(sem_rosto))
    linhas = math.ceil(len(sem_rosto) / por_linha)
    fig, axs = plt.subplots(linhas, por_linha, figsize=(3 * por_linha, 3.4 * linhas), squeeze=False)
    for ax in axs.ravel():
        ax.axis("off")
    for ax, c in zip(axs.ravel(), sorted(sem_rosto)):
        img_p = carregar_imagem(c)
        if img_p is not None:
            ax.imshow(cv2.cvtColor(img_p, cv2.COLOR_BGR2RGB))
        ax.set_title(os.path.basename(c), fontsize=9)
    plt.tight_layout()
    plt.show()

# Save the list as CSV (included in the final .zip)
os.makedirs(OUT_DIR, exist_ok=True)
pd.DataFrame(
    [{"file": os.path.relpath(c, IMG_DIR), "reason": "no face detected"} for c in sorted(sem_rosto)]
    + [{"file": os.path.relpath(c, IMG_DIR), "reason": "unreadable"} for c in sorted(com_erro)]
).to_csv(os.path.join(OUT_DIR, "problem_images.csv"), index=False)

df_ok = df_join[df_join["caminho"].isin(cache)].copy()

## 7. Generate average faces

In [ ]:
import matplotlib.pyplot as plt

os.makedirs(OUT_DIR, exist_ok=True)


def slug(s):
    s = re.sub(r"[^\w\-]+", "_", str(s), flags=re.UNICODE).strip("_")
    return s or "empty"


resultados = []   # each item: dict with group metadata + the average image

for grupo_cols in AGRUPAMENTOS:
    nome_grupo = "_".join(grupo_cols)
    sub = df_ok.dropna(subset=grupo_cols)
    for chave, g in sub.groupby(grupo_cols):
        chave = chave if isinstance(chave, tuple) else (chave,)
        n = len(g)
        rotulo = f"{nome_grupo}={'/'.join(map(str, chave))}"
        if n < MIN_IMAGENS:
            print(f"[skipped] {rotulo}: only {n} face(s) (minimum {MIN_IMAGENS})")
            continue
        imgs = [cache[c][0] for c in g["caminho"]]
        lms = [cache[c][1] for c in g["caminho"]]
        media = rosto_medio(imgs, lms)
        pasta = os.path.join(OUT_DIR, nome_grupo)
        os.makedirs(pasta, exist_ok=True)
        cv2.imwrite(os.path.join(pasta, f"{slug('_'.join(map(str, chave)))}_n{n}.jpg"), media)

        # Short display label: drops "modelo" (already in the grid title) and the "group=" prefix, e.g. "homem/Silva".
        outras_colunas = [c for c in grupo_cols if c != "modelo"]
        valores = [chave[grupo_cols.index(c)] for c in outras_colunas]
        rotulo_exibicao = "/".join(str(v) for v in valores) if valores else nome_grupo

        resultados.append({
            "grupo_cols": grupo_cols,
            "chave": chave,
            "rotulo": rotulo,
            "rotulo_exibicao": rotulo_exibicao,
            "n": n,
            "imagem": media,
            "modelo": chave[grupo_cols.index("modelo")] if "modelo" in grupo_cols else None,
        })
        print(f"[ok] {rotulo}  (n={n})")

print(f"\n{len(resultados)} average faces saved in {OUT_DIR}")

## 8. Plot one grid per model

Saves one PNG per model in `OUT_DIR`. Surname and category groups are ordered Popular first, then Elite, then the rest.

In [ ]:
# Map each surname to its category (Popular/Elite/...), only used for ordering
sobrenome_para_categoria = {}
if "categoria" in df_ok.columns and "sobrenome" in df_ok.columns:
    aux = df_ok.dropna(subset=["sobrenome", "categoria"]).drop_duplicates("sobrenome")
    sobrenome_para_categoria = dict(zip(aux["sobrenome"], aux["categoria"]))

ORDEM_CATEGORIA = {"Popular": 0, "Elite": 1}  # anything else (Controle, Ambíguo, ...) comes after


def prioridade_categoria(categoria):
    return ORDEM_CATEGORIA.get(categoria, 50)


def chave_ordenacao(item):
    grupo_cols = item["grupo_cols"]
    if "sobrenome" in grupo_cols:
        sobrenome = item["chave"][grupo_cols.index("sobrenome")]
        categoria = sobrenome_para_categoria.get(sobrenome, "")
        return (prioridade_categoria(categoria), str(sobrenome), item["rotulo"])
    if "categoria" in grupo_cols:
        categoria = item["chave"][grupo_cols.index("categoria")]
        return (prioridade_categoria(categoria), str(categoria), item["rotulo"])
    return (10, item["rotulo"])  # gender/region groups keep alphabetical order


def mostrar_e_salvar_por_modelo(resultados, por_linha=5):
    if not resultados:
        print("Nothing to show.")
        return

    modelos = sorted({item["modelo"] for item in resultados if item["modelo"] is not None})
    if not modelos:
        modelos = [None]  # fallback if "modelo" is in no grouping

    for modelo in modelos:
        subset = sorted(
            (item for item in resultados if item["modelo"] == modelo),
            key=chave_ordenacao,
        )
        if not subset:
            continue

        linhas = math.ceil(len(subset) / por_linha)
        fig, axs = plt.subplots(linhas, por_linha, figsize=(3 * por_linha, 4.2 * linhas))
        axs = np.atleast_1d(axs).ravel()
        for ax in axs:
            ax.axis("off")
        for ax, item in zip(axs, subset):
            ax.imshow(cv2.cvtColor(item["imagem"], cv2.COLOR_BGR2RGB))
            ax.set_title(f"{item['rotulo_exibicao']}\n(n={item['n']})", fontsize=9)

        titulo = f"Model: {modelo}" if modelo else "Average faces"
        fig.suptitle(titulo, fontsize=14, y=1.02)
        plt.tight_layout()

        caminho_png = os.path.join(OUT_DIR, f"grid_{slug(str(modelo))}.png")
        plt.savefig(caminho_png, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {caminho_png}")


mostrar_e_salvar_por_modelo(resultados)

## 9. Special averages (all elite faces; selected surnames by region group)

Per model: the average of all `Elite` faces and, for the surnames in `SOBRENOMES_MS`, each region of a group plus the average of the group. Edit `SOBRENOMES_MS`, `GENERO_MS`, `GRUPOS_MS` or `JUNTAR_MODELOS` below.

In [ ]:
# ---------------- Settings ----------------
JUNTAR_MODELOS = False          # False = one independent average per model; True = pool all models
SOBRENOMES_MS = ["Moreira Salles", "Safra", "Lemann"]   # surnames pooled together; a one-item list works too
NOME_MS = " + ".join(SOBRENOMES_MS)
GENERO_MS = None            # None pools homem + mulher + pessoa
GRUPOS_MS = {
    "No region + Sul + Sudeste": ["Sem menção", "Sul", "Sudeste"],
    "Nordeste + Norte + Centro-Oeste":     ["Nordeste", "Norte", "Centro-Oeste"],
}
# -----------------------------------------------


def media_de(df, **filtros):
    # Average of the faces in df matching the filters (column=value or column=[values]; None = no filter).
    # Returns (BGR image, n), or (None, 0) if there are no faces.
    sel = df
    for col, val in filtros.items():
        if val is None:
            continue
        vals = list(val) if isinstance(val, (list, tuple, set)) else [val]
        sel = sel[sel[col].isin(vals)]
    if len(sel) == 0:
        return None, 0
    imgs = [cache[c][0] for c in sel["caminho"]]
    lms = [cache[c][1] for c in sel["caminho"]]
    return rosto_medio(imgs, lms), len(sel)


pasta_esp = os.path.join(OUT_DIR, "special")
os.makedirs(pasta_esp, exist_ok=True)

modelos_loop = [None] if JUNTAR_MODELOS else sorted(df_ok["modelo"].dropna().unique())

for modelo in modelos_loop:
    base = df_ok if modelo is None else df_ok[df_ok["modelo"] == modelo]
    nome_modelo = "all_models" if modelo is None else str(modelo)

    # Each figure row: (row label, [(title, image), ...])
    img, n = media_de(base, categoria="Elite")
    linhas = [("Elite (all)", [(f"All elite\nsurnames (n={n})", img)])]
    if img is not None:
        cv2.imwrite(os.path.join(pasta_esp, f"{slug(nome_modelo)}_elite_all_n{n}.jpg"), img)

    for nome_grupo, regioes in GRUPOS_MS.items():
        paineis = []
        for reg in regioes:
            img, n = media_de(base, sobrenome=SOBRENOMES_MS, genero=GENERO_MS, regiao=reg)
            paineis.append((f"{reg}\n(n={n})", img))
        img, n = media_de(base, sobrenome=SOBRENOMES_MS, genero=GENERO_MS, regiao=regioes)
        paineis.append((f"Average of {len(regioes)}\n(n={n})", img))
        if img is not None:
            cv2.imwrite(os.path.join(pasta_esp, f"{slug(nome_modelo)}_{slug(NOME_MS)}_{slug(nome_grupo)}_n{n}.jpg"), img)
        linhas.append((nome_grupo, paineis))

    ncols = max(len(p) for _, p in linhas)
    fig, axs = plt.subplots(len(linhas), ncols, figsize=(3 * ncols, 4.2 * len(linhas)), squeeze=False)
    for ax in axs.ravel():
        ax.axis("off")
    for i, (rotulo_linha, paineis) in enumerate(linhas):
        axs[i][0].text(-0.08, 0.5, rotulo_linha, transform=axs[i][0].transAxes,
                       rotation=90, va="center", ha="right", fontsize=9)
        for j, (titulo_p, img_p) in enumerate(paineis):
            if img_p is not None:
                axs[i][j].imshow(cv2.cvtColor(img_p, cv2.COLOR_BGR2RGB))
            axs[i][j].set_title(titulo_p, fontsize=9)

    titulo = "All models" if modelo is None else f"Model: {modelo}"
    fig.suptitle(f"{titulo}  |  Elite and {NOME_MS}", fontsize=14, y=1.02)
    plt.tight_layout()
    caminho_png = os.path.join(pasta_esp, f"grid_special_{slug(nome_modelo)}.png")
    plt.savefig(caminho_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {caminho_png}")

## 10. Download everything as a .zip

In [ ]:
shutil.make_archive("/content/rostos_medios", "zip", OUT_DIR)
from google.colab import files
files.download("/content/rostos_medios.zip")